# finance / economy / rates

**Intent:** Output the economy's inflation rate (`finance_economy_rates_inflation_{low,base,high}_pct`), the risk-free (bank) rate (`finance_economy_rates_riskfree_{low,base,high}_pct`) and the real return of cash (`finance_economy_rates_real_{low,base,high}_pct`) to 2070 as a low, base and high scenario, in percent per year, as constants that satisfy real = risk-free less inflation. The risk-free rate is the floor of every `portfolio/return` scenario and the discount rate of `portfolio/withdrawal`, and `household/costs` adds its own premium to the inflation rate.

**Notes:**
- Placeholder: low, base and high are the smallest, the mean and the largest of the 5, 10, 20 and 40 year window means of each series
- Data splits: `training` the four window means of the bank rate, inflation and the real return of cash, synthetic until the `interest` table is read; `golden` the exact request that becomes `golden.json`
- One table: the three series come from the same `interest` rows and the identity real = risk-free less inflation is asserted in the Fit cell

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env` and the signature, and declare the model name, target and dataset names as constants (`model.md` 6.6).

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/economy/rates"
MODEL_NAME = "finance_economy_rates"
TARGETS = [{"kind": "served"}]
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "economy_rates_training_2026-11-05_2026-11-05"
GOLDEN = "economy_rates_golden"

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training` file only when it is absent (`model.md` 6.5). The source is synthetic, so no query runs.

In [ ]:
WINDOWS = [5, 10, 20, 40]
BANK = [2.0, 2.6, 3.6, 5.0]
INFLATION = [2.9, 2.7, 2.8, 3.24]

training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {"window_years": WINDOWS, "bank_pct": BANK, "inflation_pct": INFLATION, "net_pct": [round(bank - inflation, 4) for bank, inflation in zip(BANK, INFLATION, strict=True)]},
)
print("windows", training_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** summarise the window means of `training` only (R6) for each series, after asserting that the real return of cash is the bank rate less inflation in every window.

In [ ]:
bank, inflation, net = (training_pl[column].to_numpy() for column in ("bank_pct", "inflation_pct", "net_pct"))
assert abs(bank - inflation - net).max() < 1e-9, "the real return of cash is not the bank rate less inflation"


def scenarios(series):
    return [round(float(series.min()), 2), round(float(series.mean()), 2), round(float(series.max()), 2)]


CONSTANTS = dict(zip(OUTPUTS, [*scenarios(inflation), *scenarios(bank), *scenarios(net)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [2.7, 2.91, 3.24, 2.0, 3.3, 5.0, -0.9, 0.39, 1.76], "the fit did not recover the window summary"

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows and the project from the loaded signature, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUTS[0]: [1.0, None], "expect": ["ok", "error"]})

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic window means of the bank rate, inflation and real cash return, training split",
        [training_path, golden_path],
        "smallest, mean and largest of the window means",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped loader")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)